In [20]:
# Configure matplotlib for no-GUI mode (server environment)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [21]:
# Import necessary libraries
import numpy as np
import pandas as pd
from datasets import load_from_disk
from pathlib import Path
from collections import defaultdict, Counter
from tqdm import tqdm
import json
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [22]:
# Helper function to convert numpy types to native Python types for JSON serialization
def convert_to_native_types(obj):
    """
    Recursively convert numpy types to native Python types for JSON serialization
    """
    if isinstance(obj, dict):
        return {key: convert_to_native_types(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_native_types(item) for item in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

In [23]:
# Load datasets
data_root = '/net/scratch/k09562zs/LLM_reaction_Robot/Reaction_DataSet/processed'
output_dir = Path('../analysis_results')
output_dir.mkdir(exist_ok=True)

print("Loading datasets...")
train_dataset = load_from_disk(f'{data_root}/train')
val_dataset = load_from_disk(f'{data_root}/val')

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"\nSample structure:")
print(train_dataset[0].keys())

Loading datasets...
Train samples: 1660
Val samples: 571

Sample structure:
dict_keys(['id', 'speaker_video_path', 'speaker_audio_path', 'listener_video_path', 'listener_audio_path', 'listener_au_names', 'listener_au_prob', 'listener_au_act', 'listener_frame_idx', 'fps', 'duration', 'n_frames'])


---
## 1. AU激活频率distribution分析

**Objectives**：
- 识别High-freq AU vs Low-freq AU
- Calculate imbalance ratio
- 为损失函数设计Weight

In [24]:
def analyze_au_frequency(dataset, split_name='train'):
    """
    Analyze activation frequency for each AU
    """
    print(f"\n{'='*60}")
    print(f"分析 {split_name} 集AU频率distribution")
    print(f"{'='*60}")
    
    # Count activations for each AU
    au_activation_counts = defaultdict(int)
    au_total_frames = defaultdict(int)
    
    for sample in tqdm(dataset, desc=f"Processing {split_name}"):
        au_act = sample['listener_au_act']  # Dict[str, List[int]]
        
        for au_name, activations in au_act.items():
            au_activation_counts[au_name] += sum(activations)
            au_total_frames[au_name] += len(activations)
    
    # Calculate activation rates
    au_names = sorted(au_activation_counts.keys())
    activation_rates = {
        au: au_activation_counts[au] / au_total_frames[au] * 100
        for au in au_names
    }
    
    # Create DataFrame for analysis
    df = pd.DataFrame({
        'AU': au_names,
        'Activation_Count': [au_activation_counts[au] for au in au_names],
        'Total_Frames': [au_total_frames[au] for au in au_names],
        'Activation_Rate_%': [activation_rates[au] for au in au_names]
    })
    
    # Sort by activation rate
    df = df.sort_values('Activation_Rate_%', ascending=False)
    
    print("\nAU激活统计（按频率Sort by activation rate）：")
    print(df.to_string(index=False))
    
    # 分类AU
    high_freq = df[df['Activation_Rate_%'] > 20.0]
    mid_freq = df[(df['Activation_Rate_%'] >= 5.0) & (df['Activation_Rate_%'] <= 20.0)]
    low_freq = df[df['Activation_Rate_%'] < 5.0]
    
    print(f"\nFrequency Classification:")
    print(f"  High-freq AU (>20%): {len(high_freq)} 个 - {list(high_freq['AU'])}")
    print(f"  Mid-freq AU (5-20%): {len(mid_freq)} 个 - {list(mid_freq['AU'])}")
    print(f"  Low-freq AU (<5%): {len(low_freq)} 个 - {list(low_freq['AU'])}")
    
    # 计算不平衡比
    max_rate = df['Activation_Rate_%'].max()
    min_rate = df['Activation_Rate_%'].min()
    imbalance_ratio = max_rate / min_rate if min_rate > 0 else float('inf')
    print(f"\nImbalance Ratio (max/min): {imbalance_ratio:.2f}x")
    
    return df, activation_rates

# Analyze train and val sets
train_au_df, train_au_rates = analyze_au_frequency(train_dataset, 'train')
val_au_df, val_au_rates = analyze_au_frequency(val_dataset, 'val')


分析 train 集AU频率distribution


Processing train: 100%|██████████| 1660/1660 [00:24<00:00, 67.63it/s] 



AU激活统计（按频率Sort by activation rate）：
  AU  Activation_Count  Total_Frames  Activation_Rate_%
AU14           3048322       3438412          88.654937
AU18           2805732       3438412          81.599645
AU20           1805618       3438412          52.513137
 AU6           1609969       3438412          46.823039
AU23           1242611       3438412          36.139096
AU10           1239414       3438412          36.046117
AU26            997153       3438412          29.000393
AU17            985856       3438412          28.671840
AU15            856270       3438412          24.903066
AU12            787723       3438412          22.909500
AU25            711589       3438412          20.695280
 AU9            384474       3438412          11.181732
 AU2            349715       3438412          10.170829
 AU5            253905       3438412           7.384368
 AU7            233588       3438412           6.793485
 AU1            191525       3438412           5.570159
 AU4       

Processing val: 100%|██████████| 571/571 [00:07<00:00, 79.04it/s] 


AU激活统计（按频率Sort by activation rate）：
  AU  Activation_Count  Total_Frames  Activation_Rate_%
AU14            903739       1016264          88.927582
AU18            804597       1016264          79.172046
AU20            578978       1016264          56.971220
 AU6            464259       1016264          45.682913
AU23            449243       1016264          44.205344
AU10            385499       1016264          37.932958
AU26            322283       1016264          31.712527
AU15            316282       1016264          31.122031
AU17            314169       1016264          30.914113
AU12            221590       1016264          21.804374
AU25            203577       1016264          20.031901
 AU9            136119       1016264          13.394059
 AU2             83628       1016264           8.228964
 AU7             79260       1016264           7.799155
 AU5             59161       1016264           5.821420
 AU1             47918       1016264           4.715113
 AU4       

In [25]:
# Visualize AU activation frequency
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Train Set
ax1 = axes[0]
bars1 = ax1.bar(range(len(train_au_df)), train_au_df['Activation_Rate_%'], 
                color=['red' if x < 5 else 'orange' if x < 20 else 'green' 
                       for x in train_au_df['Activation_Rate_%']])
ax1.set_xticks(range(len(train_au_df)))
ax1.set_xticklabels(train_au_df['AU'], rotation=45, ha='right')
ax1.set_ylabel('Activation Rate (%)')
ax1.set_title(f'Train Set: AU Activation Frequency (N={len(train_dataset)} samples)')
ax1.axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='Low freq threshold (5%)')
ax1.axhline(y=20, color='green', linestyle='--', alpha=0.5, label='High freq threshold (20%)')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Val Set
ax2 = axes[1]
bars2 = ax2.bar(range(len(val_au_df)), val_au_df['Activation_Rate_%'],
                color=['red' if x < 5 else 'orange' if x < 20 else 'green' 
                       for x in val_au_df['Activation_Rate_%']])
ax2.set_xticks(range(len(val_au_df)))
ax2.set_xticklabels(val_au_df['AU'], rotation=45, ha='right')
ax2.set_ylabel('Activation Rate (%)')
ax2.set_title(f'Val Set: AU Activation Frequency (N={len(val_dataset)} samples)')
ax2.axhline(y=5, color='orange', linestyle='--', alpha=0.5)
ax2.axhline(y=20, color='green', linestyle='--', alpha=0.5)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
output_path = output_dir / 'au_activation_frequency.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved figure: {output_path}")
plt.close()


✓ Saved figure: ../analysis_results/au_activation_frequency.png


In [26]:
# Calculate recommended loss weights (based on inverse frequency)
def calculate_loss_weights(activation_rates):
    """
    基于Activation Rate计算损失Weight：Low-freq AU获得更高Weight
    """
    weights = {}
    rates = np.array(list(activation_rates.values()))
    
    # Method 1: Inverse frequency (with smoothing)
    for au, rate in activation_rates.items():
        weights[au] = 1.0 / np.log(rate + 1.0)  # log smoothing to avoid extreme values
    
    # 归一化到[0.5, 2.0]范围
    min_w = min(weights.values())
    max_w = max(weights.values())
    weights_normalized = {
        au: 0.5 + 1.5 * (w - min_w) / (max_w - min_w)
        for au, w in weights.items()
    }
    
    return weights_normalized

loss_weights = calculate_loss_weights(train_au_rates)

print("\nRecommended AU Loss Weights (for Focal Loss or Weighted BCE):")
print("-" * 60)
for au in sorted(loss_weights.keys()):
    rate = train_au_rates[au]
    weight = loss_weights[au]
    print(f"{au:6s}: Activation Rate={rate:5.2f}% → Weight={weight:.3f}")

# 保存Weight配置
weight_config = {
    'au_loss_weights': loss_weights,
    'au_activation_rates': train_au_rates,
    'description': 'AU loss weights for training (higher weight for rare AUs)'
}
with open(output_dir / 'au_loss_weights.json', 'w') as f:
    json.dump(weight_config, f, indent=2)
print(f"\n✓ 保存Weight配置: {output_dir / 'au_loss_weights.json'}")


Recommended AU Loss Weights (for Focal Loss or Weighted BCE):
------------------------------------------------------------
AU1   : Activation Rate= 5.57% → Weight=1.313
AU10  : Activation Rate=36.05% → Weight=0.643
AU12  : Activation Rate=22.91% → Weight=0.744
AU14  : Activation Rate=88.65% → Weight=0.500
AU15  : Activation Rate=24.90% → Weight=0.724
AU17  : Activation Rate=28.67% → Weight=0.691
AU18  : Activation Rate=81.60% → Weight=0.511
AU2   : Activation Rate=10.17% → Weight=1.006
AU20  : Activation Rate=52.51% → Weight=0.576
AU23  : Activation Rate=36.14% → Weight=0.643
AU25  : Activation Rate=20.70% → Weight=0.770
AU26  : Activation Rate=29.00% → Weight=0.689
AU4   : Activation Rate= 2.54% → Weight=2.000
AU5   : Activation Rate= 7.38% → Weight=1.153
AU6   : Activation Rate=46.82% → Weight=0.595
AU7   : Activation Rate= 6.79% → Weight=1.197
AU9   : Activation Rate=11.18% → Weight=0.968

✓ 保存Weight配置: ../analysis_results/au_loss_weights.json


---
## 2. Speaker-Listener Delay Pattern Analysis

**Objectives**：
- Discover natural listener reaction delays
- 为time offset训练提供数据依据
- Different AUs may have different reaction delays

In [27]:
def analyze_reaction_delay(dataset, split_name='train', sample_size=200):
    """
    Analyze speaker-listener reaction delay patterns
    
    Method: Calculate listener AU activation time delay relative to speaker video start
    （This is simplified analysis, complete version needs to parse speaker action timestamps）
    """
    print(f"\n{'='*60}")
    print(f"分析 {split_name} 集的Reaction Delay Pattern")
    print(f"{'='*60}")
    
    au_onset_delays = defaultdict(list)  # Delay of first activation for each AU
    au_change_intervals = defaultdict(list)  # Intervals between AU state changes
    
    # Sample analysis (full analysis would be slow)
    sample_indices = np.random.choice(
        len(dataset), 
        min(sample_size, len(dataset)), 
        replace=False
    )
    
    for idx in tqdm(sample_indices, desc=f"Analyzing {split_name} delays"):
        sample = dataset[int(idx)]
        au_act = sample['listener_au_act']
        fps = sample['fps']
        
        for au_name, activations in au_act.items():
            activations = np.array(activations)
            
            # Find first activation frame
            first_activation = np.where(activations == 1)[0]
            if len(first_activation) > 0:
                onset_frame = first_activation[0]
                onset_time_ms = (onset_frame / fps) * 1000
                au_onset_delays[au_name].append(onset_time_ms)
            
            # Find state change points (0→1 or 1→0)
            changes = np.diff(activations)
            change_frames = np.where(changes != 0)[0]
            
            if len(change_frames) > 1:
                intervals = np.diff(change_frames)
                intervals_ms = (intervals / fps) * 1000
                au_change_intervals[au_name].extend(intervals_ms)
    
    # Statistical results
    delay_stats = {}
    for au in sorted(au_onset_delays.keys()):
        delays = au_onset_delays[au]
        if len(delays) > 0:
            delay_stats[au] = {
                'mean_onset_ms': np.mean(delays),
                'median_onset_ms': np.median(delays),
                'std_onset_ms': np.std(delays),
                'n_samples': len(delays)
            }
    
    print("\nAU First Activation Delay Statistics (relative to video start):")
    print("-" * 80)
    print(f"{'AU':<8} {'Mean(ms)':<12} {'Median(ms)':<12} {'Std Dev(ms)':<12} {'N Samples':<10}")
    print("-" * 80)
    for au in sorted(delay_stats.keys()):
        stats = delay_stats[au]
        print(f"{au:<8} {stats['mean_onset_ms']:<12.1f} {stats['median_onset_ms']:<12.1f} "
              f"{stats['std_onset_ms']:<12.1f} {stats['n_samples']:<10}")
    
    return delay_stats, au_onset_delays, au_change_intervals

# 分析Train Set
train_delay_stats, train_onset_delays, train_intervals = analyze_reaction_delay(
    train_dataset, 'train', sample_size=300
)


分析 train 集的Reaction Delay Pattern


Analyzing train delays: 100%|██████████| 300/300 [00:04<00:00, 67.43it/s]


AU First Activation Delay Statistics (relative to video start):
--------------------------------------------------------------------------------
AU       Mean(ms)     Median(ms)   Std Dev(ms)  N Samples 
--------------------------------------------------------------------------------
AU1      20927.8      11666.7      25447.2      158       
AU10     8103.3       266.7        17607.7      280       
AU12     9686.3       700.0        17570.3      258       
AU14     190.2        0.0          1044.6       298       
AU15     7406.6       1466.7       13455.8      261       
AU17     5920.7       700.0        12724.9      270       
AU18     955.6        0.0          5358.3       294       
AU2      14601.6      7133.3       19307.3      162       
AU20     2789.9       0.0          8437.5       290       
AU23     7107.5       866.7        17101.5      279       
AU25     11580.2      5933.3       14902.3      263       
AU26     8576.4       1933.3       14970.6      265       
AU4   

In [28]:
# Visualize delay distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. AU First Activation Delay Box Plot
ax1 = axes[0, 0]
au_names_sorted = sorted(train_onset_delays.keys())
delays_data = [train_onset_delays[au] for au in au_names_sorted]
bp = ax1.boxplot(delays_data, labels=au_names_sorted, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
ax1.set_xlabel('AU')
ax1.set_ylabel('Onset Delay (ms)')
ax1.set_title('AU First Activation Delay Distribution (relative to video start)')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# 2. Mean延迟的条形图
ax2 = axes[0, 1]
mean_delays = [train_delay_stats[au]['mean_onset_ms'] for au in au_names_sorted]
bars = ax2.barh(range(len(au_names_sorted)), mean_delays, color='steelblue')
ax2.set_yticks(range(len(au_names_sorted)))
ax2.set_yticklabels(au_names_sorted)
ax2.set_xlabel('Mean Onset Delay (ms)')
ax2.set_title('各AU的Mean首次激活延迟')
ax2.axvline(x=200, color='red', linestyle='--', alpha=0.5, label='200msReference line')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

# 3. Global Delay Distribution Histogram
ax3 = axes[1, 0]
all_delays = []
for delays in train_onset_delays.values():
    all_delays.extend(delays)
ax3.hist(all_delays, bins=50, color='seagreen', alpha=0.7, edgecolor='black')
ax3.axvline(x=np.median(all_delays), color='red', linestyle='--', 
            label=f'Median: {np.median(all_delays):.0f}ms')
ax3.axvline(x=np.mean(all_delays), color='orange', linestyle='--',
            label=f'Mean: {np.mean(all_delays):.0f}ms')
ax3.set_xlabel('Onset Delay (ms)')
ax3.set_ylabel('Frequency')
ax3.set_title('First Activation Delay Distribution for All AUs')
ax3.legend()
ax3.grid(alpha=0.3)

# 4. Recommended Time Offset
ax4 = axes[1, 1]
ax4.axis('off')

# 计算建议值
global_median = np.median(all_delays)
global_mean = np.mean(all_delays)
fps = 30
recommended_frames = int(global_median / (1000 / fps))

recommendation_text = f"""
📊 Delay Analysis Results and Recommendations

Global Statistics:
  • Median延迟: {global_median:.1f} ms
  • Mean延迟: {global_mean:.1f} ms
  • Std Dev: {np.std(all_delays):.1f} ms

🎯 Training Recommendations:

1. Time Offset Setting (τ):
   Recommended: τ = {recommended_frames} frames ≈ {global_median:.0f} ms
   
   Training Code:
   target_frame = current_frame + {recommended_frames}

2. Strategy Comparison:
   • No offset(τ=0): Learn synchronous reaction (unnatural)
   • Fixed offset(τ={recommended_frames}): 学习Mean延迟
   • Adaptive offset: Different τ for each AU

3. Robot Deployment:
   • Predict future {global_median:.0f}ms AU
   • 可补偿网络+机械延迟
"""

ax4.text(0.05, 0.95, recommendation_text, 
         transform=ax4.transAxes,
         fontsize=10,
         verticalalignment='top',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
output_path = output_dir / 'reaction_delay_analysis.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved figure: {output_path}")
plt.close()

# Saved delay config
delay_config = convert_to_native_types({
    'recommended_delay_ms': float(global_median),
    'recommended_delay_frames': int(recommended_frames),
    'fps': fps,
    'global_statistics': {
        'median_ms': float(global_median),
        'mean_ms': float(global_mean),
        'std_ms': float(np.std(all_delays))
    },
    'per_au_delays': train_delay_stats
})
with open(output_dir / 'reaction_delay_config.json', 'w') as f:
    json.dump(delay_config, f, indent=2)
print(f"✓ Saved delay config: {output_dir / 'reaction_delay_config.json'}")

/tmp/ipykernel_3401652/2991910136.py:8: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax1.boxplot(delays_data, labels=au_names_sorted, patch_artist=True)
/tmp/ipykernel_3401652/2991910136.py:88: UserWarning: Glyph 21508 (\N{CJK UNIFIED IDEOGRAPH-5404}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3401652/2991910136.py:88: UserWarning: Glyph 30340 (\N{CJK UNIFIED IDEOGRAPH-7684}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3401652/2991910136.py:88: UserWarning: Glyph 39318 (\N{CJK UNIFIED IDEOGRAPH-9996}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3401652/2991910136.py:88: UserWarning: Glyph 27425 (\N{CJK UNIFIED IDEOGRAPH-6B21}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3401652/2991910136.py:88: UserWarning: Glyph 28608 (\N{CJK UNIFIED IDEOGRAPH-6FC0}) m


✓ Saved figure: ../analysis_results/reaction_delay_analysis.png
✓ Saved delay config: ../analysis_results/reaction_delay_config.json


---
## 3. samples长度distribution分析

**Objectives**：
- 了解sequence lengthdistribution
- Optimize batch size and max_seq_length
- 设计训练window策略

In [29]:
def analyze_sample_length(dataset, split_name='train'):
    """
    Analyze sample length distribution
    """
    print(f"\n{'='*60}")
    print(f"分析 {split_name} 集的samples长度distribution")
    print(f"{'='*60}")
    
    n_frames_list = []
    durations_list = []
    fps_list = []
    
    for sample in tqdm(dataset, desc=f"Processing {split_name}"):
        n_frames_list.append(sample['n_frames'])
        durations_list.append(sample['duration'])
        fps_list.append(sample['fps'])
    
    n_frames = np.array(n_frames_list)
    durations = np.array(durations_list)
    fps_mean = np.mean(fps_list)
    
    # 统计
    stats = {
        'count': len(n_frames),
        'mean_frames': np.mean(n_frames),
        'median_frames': np.median(n_frames),
        'std_frames': np.std(n_frames),
        'min_frames': np.min(n_frames),
        'max_frames': np.max(n_frames),
        'p25_frames': np.percentile(n_frames, 25),
        'p75_frames': np.percentile(n_frames, 75),
        'p90_frames': np.percentile(n_frames, 90),
        'p95_frames': np.percentile(n_frames, 95),
        'p99_frames': np.percentile(n_frames, 99),
        'mean_duration_sec': np.mean(durations),
        'fps': fps_mean
    }
    
    print(f"\nN Samples量: {stats['count']}")
    print(f"\nframes数统计:")
    print(f"  Mean: {stats['mean_frames']:.1f} frames ({stats['mean_duration_sec']:.1f} sec)")
    print(f"  Median: {stats['median_frames']:.0f} frames")
    print(f"  Std Dev: {stats['std_frames']:.1f} frames")
    print(f"  Min: {stats['min_frames']} frames")
    print(f"  Max: {stats['max_frames']} frames")
    print(f"\nPercentiles:")
    print(f"  25%: {stats['p25_frames']:.0f} frames")
    print(f"  50%: {stats['median_frames']:.0f} frames")
    print(f"  75%: {stats['p75_frames']:.0f} frames")
    print(f"  90%: {stats['p90_frames']:.0f} frames")
    print(f"  95%: {stats['p95_frames']:.0f} frames")
    print(f"  99%: {stats['p99_frames']:.0f} frames")
    
    return stats, n_frames, durations

# Analyze both sets
train_length_stats, train_n_frames, train_durations = analyze_sample_length(train_dataset, 'train')
val_length_stats, val_n_frames, val_durations = analyze_sample_length(val_dataset, 'val')


分析 train 集的samples长度distribution


Processing train: 100%|██████████| 1660/1660 [00:23<00:00, 72.11it/s] 



N Samples量: 1660

frames数统计:
  Mean: 2071.3 frames (69.0 sec)
  Median: 1626 frames
  Std Dev: 1562.3 frames
  Min: 58 frames
  Max: 21145 frames

Percentiles:
  25%: 902 frames
  50%: 1626 frames
  75%: 2940 frames
  90%: 4375 frames
  95%: 4558 frames
  99%: 6328 frames

分析 val 集的samples长度distribution


Processing val: 100%|██████████| 571/571 [00:06<00:00, 84.99it/s] 


N Samples量: 571

frames数统计:
  Mean: 1779.8 frames (59.3 sec)
  Median: 1439 frames
  Std Dev: 1321.6 frames
  Min: 245 frames
  Max: 10464 frames

Percentiles:
  25%: 872 frames
  50%: 1439 frames
  75%: 2190 frames
  90%: 3607 frames
  95%: 4516 frames
  99%: 6186 frames


In [30]:
# Visualize length distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. frames数distribution直方图
ax1 = axes[0, 0]
ax1.hist(train_n_frames, bins=50, alpha=0.7, label='Train', color='blue', edgecolor='black')
ax1.hist(val_n_frames, bins=50, alpha=0.7, label='Val', color='orange', edgecolor='black')
ax1.axvline(train_length_stats['mean_frames'], color='blue', linestyle='--', 
            label=f"Train Mean: {train_length_stats['mean_frames']:.0f}")
ax1.axvline(val_length_stats['mean_frames'], color='orange', linestyle='--',
            label=f"Val Mean: {val_length_stats['mean_frames']:.0f}")
ax1.set_xlabel('Number of Frames')
ax1.set_ylabel('Frequency')
ax1.set_title('samplesframes数distribution')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Cumulative Distribution Function (CDF)
ax2 = axes[0, 1]
sorted_train = np.sort(train_n_frames)
sorted_val = np.sort(val_n_frames)
ax2.plot(sorted_train, np.arange(len(sorted_train)) / len(sorted_train) * 100, 
         label='Train', color='blue', linewidth=2)
ax2.plot(sorted_val, np.arange(len(sorted_val)) / len(sorted_val) * 100,
         label='Val', color='orange', linewidth=2)
ax2.axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% coverage')
ax2.axhline(y=95, color='green', linestyle='--', alpha=0.5, label='95% coverage')
ax2.set_xlabel('Number of Frames')
ax2.set_ylabel('Cumulative Percentage (%)')
ax2.set_title('Cumulative Distribution Function (CDF)')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Box Plot Comparison
ax3 = axes[1, 0]
bp = ax3.boxplot([train_n_frames, val_n_frames], 
                  labels=['Train', 'Val'],
                  patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightcoral')
ax3.set_ylabel('Number of Frames')
ax3.set_title('frames数distribution箱线图')
ax3.grid(axis='y', alpha=0.3)

# 4. Recommended Configuration
ax4 = axes[1, 1]
ax4.axis('off')

# Calculate recommended max_seq_length
p90 = train_length_stats['p90_frames']
p95 = train_length_stats['p95_frames']
p99 = train_length_stats['p99_frames']
mean = train_length_stats['mean_frames']

recommendation_text = f"""
📊 Sample Length Analysis Results and Recommendations

Statistical Summary (Train):
  • Mean长度: {mean:.0f} frames ({mean/30:.1f} sec)
  • 90%samples ≤ {p90:.0f} frames
  • 95%samples ≤ {p95:.0f} frames
  • 99%samples ≤ {p99:.0f} frames

🎯 Training Configuration Recommendations:

1. Max Sequence Length:
   • Option A (90% coverage): {int(p90)}
   • Option B (95% coverage): {int(p95)} ← Recommended
   • Option C (99% coverage): {int(p99)}

2. Memory Optimization Strategy:
   If choosing Option B({int(p95)}frames):
   • Can save {(1 - p95/p99)*100:.1f}% GPU memory
   • Only discard 5% 最长samples
   • 或对长samples裁剪/滑动window

3. Sliding Window Training:
   Window size: 1000-1500 frames (33-50sec)
   Stride: 500 frames (overlapping training)
   
4. Batch SizeEstimation:
   Assuminghidden_dim=256:
   • max_len={int(p95)}: batch_size ≈ 8-16
   • max_len=1000: batch_size ≈ 32-64
"""

ax4.text(0.05, 0.95, recommendation_text,
         transform=ax4.transAxes,
         fontsize=9,
         verticalalignment='top',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

plt.tight_layout()
output_path = output_dir / 'sample_length_distribution.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved figure: {output_path}")
plt.close()

# Saved length config
length_config = convert_to_native_types({
    'train_statistics': train_length_stats,
    'val_statistics': val_length_stats,
    'recommendations': {
        'max_seq_length_90pct': int(p90),
        'max_seq_length_95pct': int(p95),
        'max_seq_length_99pct': int(p99),
        'recommended': int(p95),
        'sliding_window_size': 1000,
        'sliding_window_stride': 500
    }
})
with open(output_dir / 'sample_length_config.json', 'w') as f:
    json.dump(length_config, f, indent=2)
print(f"✓ Saved length config: {output_dir / 'sample_length_config.json'}")

/tmp/ipykernel_3401652/1629156040.py:36: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax3.boxplot([train_n_frames, val_n_frames],
/tmp/ipykernel_3401652/1629156040.py:94: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3401652/1629156040.py:94: UserWarning: Glyph 31665 (\N{CJK UNIFIED IDEOGRAPH-7BB1}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3401652/1629156040.py:94: UserWarning: Glyph 32447 (\N{CJK UNIFIED IDEOGRAPH-7EBF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3401652/1629156040.py:94: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3401652/1629156040.py:94: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans Mono.
 


✓ Saved figure: ../analysis_results/sample_length_distribution.png
✓ Saved length config: ../analysis_results/sample_length_config.json


---
## 4. Data QualityCheck

**Objectives**：
- Check文件完整性
- Discover outliers and noise
- CheckData leakage

In [31]:
def check_data_quality(dataset, split_name='train'):
    """
    Comprehensive data quality check
    """
    print(f"\n{'='*60}")
    print(f"Check {split_name} set data quality")
    print(f"{'='*60}")
    
    issues = {
        'missing_files': [],
        'invalid_au_prob': [],
        'frame_mismatch': [],
        'fps_inconsistent': [],
        'short_samples': [],
        'noisy_au': []
    }
    
    fps_values = []
    
    for idx, sample in enumerate(tqdm(dataset, desc=f"Quality check {split_name}")):
        sample_id = sample['id']
        
        # 1. Check文件存在性
        for key in ['speaker_video_path', 'speaker_audio_path', 
                    'listener_video_path', 'listener_audio_path']:
            if not os.path.exists(sample[key]):
                issues['missing_files'].append((idx, sample_id, key))
        
        # 2. CheckAU概率范围
        au_prob = sample['listener_au_prob']
        for au_name, probs in au_prob.items():
            probs_arr = np.array(probs)
            if np.any(probs_arr < 0) or np.any(probs_arr > 1):
                issues['invalid_au_prob'].append((idx, sample_id, au_name))
        
        # 3. Checkframes数consistency
        n_frames = sample['n_frames']
        duration = sample['duration']
        fps = sample['fps']
        expected_frames = int(duration * fps)
        
        if abs(n_frames - expected_frames) > fps * 0.1:  # 容差±0.1sec
            issues['frame_mismatch'].append((
                idx, sample_id, n_frames, expected_frames
            ))
        
        # 4. FPSconsistency
        fps_values.append(fps)
        if abs(fps - 30.0) > 0.1:  # should all be 30 FPS
            issues['fps_inconsistent'].append((idx, sample_id, fps))
        
        # 5. Check过短samples
        if n_frames < 30:  # <1sec
            issues['short_samples'].append((idx, sample_id, n_frames))
        
        # 6. CheckAU标注噪声（抖动）
        au_act = sample['listener_au_act']
        for au_name, activations in au_act.items():
            act_arr = np.array(activations)
            if len(act_arr) > 10:
                # Calculate change frequency
                changes = np.sum(np.abs(np.diff(act_arr)))
                change_rate = changes / len(act_arr)
                if change_rate > 0.3:  # >30%的frames在变化（可能太抖）
                    issues['noisy_au'].append((
                        idx, sample_id, au_name, change_rate
                    ))
    
    # Report results
    print(f"\n质量Check结果:")
    print("-" * 60)
    
    for issue_type, issue_list in issues.items():
        count = len(issue_list)
        percentage = count / len(dataset) * 100
        status = "✓" if count == 0 else "⚠"
        print(f"{status} {issue_type:20s}: {count:4d} ({percentage:5.2f}%)")
        
        # Show first 3 examples
        if count > 0 and count <= 5:
            for item in issue_list:
                print(f"    → {item}")
        elif count > 5:
            for item in issue_list[:3]:
                print(f"    → {item}")
            print(f"    ... and {count-3} more")
    
    # FPS统计
    fps_unique = np.unique(fps_values)
    print(f"\nFPSdistribution: {fps_unique}")
    
    return issues

# Check两个集合
train_issues = check_data_quality(train_dataset, 'train')
val_issues = check_data_quality(val_dataset, 'val')


Check train set data quality


Quality check train: 100%|██████████| 1660/1660 [01:05<00:00, 25.29it/s]



质量Check结果:
------------------------------------------------------------
✓ missing_files       :    0 ( 0.00%)
✓ invalid_au_prob     :    0 ( 0.00%)
⚠ frame_mismatch      :    1 ( 0.06%)
    → (1231, 'Camera-2024-07-31-102814-102840', 4589, 4595)
✓ fps_inconsistent    :    0 ( 0.00%)
✓ short_samples       :    0 ( 0.00%)
✓ noisy_au            :    0 ( 0.00%)

FPSdistribution: [30.]

Check val set data quality


Quality check val: 100%|██████████| 571/571 [00:23<00:00, 24.11it/s]


质量Check结果:
------------------------------------------------------------
✓ missing_files       :    0 ( 0.00%)
✓ invalid_au_prob     :    0 ( 0.00%)
✓ frame_mismatch      :    0 ( 0.00%)
✓ fps_inconsistent    :    0 ( 0.00%)
✓ short_samples       :    0 ( 0.00%)
✓ noisy_au            :    0 ( 0.00%)

FPSdistribution: [30.]


In [32]:
# Checktrain和val之间的Data leakage
print(f"\n{'='*60}")
print("CheckData leakage（Train-Val重叠）")
print(f"{'='*60}")

train_ids = set([sample['id'] for sample in train_dataset])
val_ids = set([sample['id'] for sample in val_dataset])

overlap = train_ids & val_ids

if len(overlap) > 0:
    print(f"⚠ Warning: Found {len(overlap)} 个重复samplesID！")
    print(f"Duplicate ID examples: {list(overlap)[:5]}")
    print("\nThis causes data leakage, inflating validation metrics. Recommend cleaning data.")
else:
    print("✓ 无重复samples，train和val完全分离。")

print(f"\nTrain unique IDs: {len(train_ids)}")
print(f"Val unique IDs: {len(val_ids)}")
print(f"Overlap: {len(overlap)}")


CheckData leakage（Train-Val重叠）
⚠ Warning: Found 101 个重复samplesID！
Duplicate ID examples: ['Camera-2024-07-25-155256-155217', 'Camera-2024-07-29-135410-135329', 'Camera-2024-10-22-153434-153503', 'Camera-2024-06-26-105024-105025', 'Camera-2024-10-14-135007-134930']

This causes data leakage, inflating validation metrics. Recommend cleaning data.

Train unique IDs: 102
Val unique IDs: 115
Overlap: 101


In [33]:
# Generate data quality report
quality_report = convert_to_native_types({
    'train': {
        'total_samples': len(train_dataset),
        'issues': {k: len(v) for k, v in train_issues.items()}
    },
    'val': {
        'total_samples': len(val_dataset),
        'issues': {k: len(v) for k, v in val_issues.items()}
    },
    'data_leakage': {
        'overlapping_samples': len(overlap),
        'overlap_percentage': len(overlap) / len(train_ids) * 100 if len(train_ids) > 0 else 0
    }
})

with open(output_dir / 'data_quality_report.json', 'w') as f:
    json.dump(quality_report, f, indent=2)
print(f"\n✓ Saved quality report: {output_dir / 'data_quality_report.json'}")


✓ Saved quality report: ../analysis_results/data_quality_report.json


---
## Summary and Recommendations

Based on the above analysis, generate final configuration recommendations

In [34]:
# Generate comprehensive recommendation report
print(f"\n{'='*70}")
print("Data analysis complete! Comprehensive recommendations:")
print(f"{'='*70}\n")

summary_report = f"""
📊 Dataset Analysis Summary Report
Generated at: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

{'='*70}
1. Dataset Size
{'='*70}
  Train: {len(train_dataset)} samples
  Val:   {len(val_dataset)} samples
  
{'='*70}
2. AU Activation Frequency (Key Findings)
{'='*70}
  High-freq AU (>20%): {len([x for x in train_au_rates.values() if x > 20])} 个
  Mid-freq AU (5-20%): {len([x for x in train_au_rates.values() if 5 <= x <= 20])} 个
  Low-freq AU (<5%): {len([x for x in train_au_rates.values() if x < 5])} 个
  
  不平衡比: {max(train_au_rates.values()) / min(train_au_rates.values()):.1f}x
  
  → Recommendation: Use weighted loss function (config saved to au_loss_weights.json)

{'='*70}
3. Reaction Delay Pattern
{'='*70}
  Median延迟: {delay_config['recommended_delay_ms']:.0f} ms
  Recommendedframes偏移: {delay_config['recommended_delay_frames']} frames (30 FPS)
  
  → Recommendation: Use during training target_frame = current_frame + {delay_config['recommended_delay_frames']}
  → 机器人部署: Predict future {delay_config['recommended_delay_ms']:.0f}ms AU

{'='*70}
4. samples长度distribution
{'='*70}
  Mean长度: {train_length_stats['mean_frames']:.0f} frames ({train_length_stats['mean_duration_sec']:.1f} sec)
  95%samples ≤ {train_length_stats['p95_frames']:.0f} frames
  
  → Recommended max_seq_length: {int(train_length_stats['p95_frames'])} (覆盖95%samples)
  → or use sliding window: 1000-1500frames/window

{'='*70}
5. Data Quality
{'='*70}
  Missing files: {len(train_issues['missing_files'])} / {len(train_dataset)}
  Invalid AU probs: {len(train_issues['invalid_au_prob'])} / {len(train_dataset)}
  Data leakage: {len(overlap)} 重复samples
  
  → Status: {'Need to clean data' if len(overlap) > 0 or len(train_issues['missing_files']) > 0 else 'Data Quality良好 ✓'}

{'='*70}
6. Next Actions
{'='*70}
  ✓ View generated figures: {output_dir}/
  ✓ Use config files:
     - au_loss_weights.json (损失Weight)
     - reaction_delay_config.json (time offset)
     - sample_length_config.json (sequence length)
  
  → Before starting training:
     1. If data issues exist, clean first
     2. Adjust batch_size based on max_seq_length
     3. 在训练脚本中应用AUWeight和延迟偏移

{'='*70}
"""

print(summary_report)

# Saved summary report
with open(output_dir / 'analysis_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary_report)
print(f"\n✓ Saved summary report: {output_dir / 'analysis_summary.txt'}")

print(f"\n{'='*70}")
print("All analysis complete! Generated files:")
print(f"{'='*70}")
for file in sorted(output_dir.glob('*')):
    print(f"  • {file.name}")
print()


Data analysis complete! Comprehensive recommendations:


📊 Dataset Analysis Summary Report
Generated at: 2026-01-02 16:52:45

1. Dataset Size
  Train: 1660 samples
  Val:   571 samples

2. AU Activation Frequency (Key Findings)
  High-freq AU (>20%): 11 个
  Mid-freq AU (5-20%): 5 个
  Low-freq AU (<5%): 1 个

  不平衡比: 35.0x

  → Recommendation: Use weighted loss function (config saved to au_loss_weights.json)

3. Reaction Delay Pattern
  Median延迟: 800 ms
  Recommendedframes偏移: 24 frames (30 FPS)

  → Recommendation: Use during training target_frame = current_frame + 24
  → 机器人部署: Predict future 800ms AU

4. samples长度distribution
  Mean长度: 2071 frames (69.0 sec)
  95%samples ≤ 4558 frames

  → Recommended max_seq_length: 4558 (覆盖95%samples)
  → or use sliding window: 1000-1500frames/window

5. Data Quality
  Missing files: 0 / 1660
  Invalid AU probs: 0 / 1660
  Data leakage: 101 重复samples

  → Status: Need to clean data

6. Next Actions
  ✓ View generated figures: ../analysis_results/
  